# 01 — Build Player Season Dataset

This notebook constructs the base modelling dataset: one row per player per season, with performance statistics and end-of-season market valuations.

**Pipeline overview:**

```
Raw Transfermarkt tables
        ↓
Join appearances + games
        ↓
Aggregate to player-season
        ↓
Calculate goals/90, assists/90
        ↓
Join player information
        ↓
Calculate age
        ↓
Attach historical valuation
        ↓
Save → data/interim/player_season_base.parquet
```

**Key tables used:**
| Table | Description |
|---|---|
| `appearances` | One row per player per match — game stats live here |
| `games` | Match metadata including the season each game belongs to |
| `players` | One row per player — name, position, date of birth |
| `player_valuations` | Historical market value snapshots for each player |

## 1. Setup

Connect to the DuckDB database and confirm the available tables.

In [6]:
import duckdb
import pandas as pd

# Connect to the local database file
con = duckdb.connect("../data/raw/transfermarkt-datasets.duckdb")

# List all tables in the database
con.sql("SHOW TABLES").df()

,name
0,appearances
1,club_games
2,clubs
3,competitions
4,countries
5,game_events
6,game_lineups
7,games
8,national_teams
9,player_valuations


## 2. Data Exploration

Quick preview of the four tables we'll be working with to understand their structure and key columns.

In [ ]:
con.sql("SELECT * FROM players LIMIT 5").df()

In [ ]:
con.sql("SELECT * FROM appearances LIMIT 5").df()

In [ ]:
con.sql("SELECT * FROM games LIMIT 5").df()

In [8]:
con.sql("SELECT * FROM player_valuations LIMIT 5").df()

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1
3,6893,2003-12-15,900000,Galatasaray,984,GB1
4,10,2004-10-04,7000000,SV Werder Bremen,398,IT1


## 3. Build End-of-Season Valuations

The `player_valuations` table contains point-in-time market value snapshots throughout the year. We need a single valuation per player per season.

**Approach:** for each player-season, take the latest valuation recorded before June 30 of that season's end year. This represents the market's assessment of the player after a full season of football.

Season convention: a season labelled `2022` runs from ~July 2022 to June 2023, so the season end date is `2023-06-30`.

In [ ]:
# Load data
player_valuations = con.sql("SELECT * FROM player_valuations").df()

# Get the season end date for each player+season window
player_valuations["season_end"] = pd.to_datetime(
    player_valuations["date"].dt.year.where(
        player_valuations["date"].dt.month >= 7,
        player_valuations["date"].dt.year - 1
    ).astype(str) + "-06-30"
)

# Add a season column to valuations (season = year the season started)
player_valuations["season"] = player_valuations["date"].dt.year.where(
    player_valuations["date"].dt.month >= 7,
    player_valuations["date"].dt.year - 1
)

# Sort and take the last valuation within each player+season window
end_of_season_value = (
    player_valuations
    .sort_values("date")
    .groupby(["player_id", "season"])
    .last()  # latest record within the season window
    .reset_index()
    [["player_id", "season", "season_end", "market_value_in_eur"]]
)

# Sanity check — Bukayo Saka (player_id 433177)
end_of_season_value[end_of_season_value['player_id'] == 433177]

,player_id,season,season_end,market_value_in_eur
238296,433177,2019,2019-06-30,20000000
238297,433177,2020,2020-06-30,65000000
238298,433177,2021,2021-06-30,65000000
238299,433177,2022,2022-06-30,120000000
238300,433177,2023,2023-06-30,140000000
238301,433177,2024,2024-06-30,150000000
238302,433177,2025,2025-06-30,110000000


## 4. Build Per-Season Player Statistics

The `appearances` table has one row per match appearance but no season label. We join it to `games` on `game_id` to get the season, then aggregate per player per season.

**Note:** `game_id` is stored as `int32` in `appearances` but as a string in `games` — we cast before merging to avoid a type mismatch error.

**Features computed:**
- `minutes`, `goals`, `assists` — raw totals for the season
- `goals_per_90`, `assists_per_90` — rate stats normalised to 90 minutes
- `age` — player age at the end of the season (June 30)

In [ ]:
# Load data
appearances = con.sql("SELECT * FROM appearances").df()
players = con.sql("SELECT * FROM players").df()
games = con.sql("SELECT * from games").df()

# game_id is an int in appearances but str in games, they need to match before merging
games["game_id"] = games["game_id"].astype("int32")

# Add season to appearances by joining on game_id
appearances_2 = appearances.merge(games, on="game_id", how="inner")
season_stats = (
    appearances_2
    .groupby(["player_id", "season"])
    .agg(
        minutes=("minutes_played", "sum"),
        goals=("goals", "sum"),
        assists=("assists", "sum")
    )
)
season_stats["goals_per_90"] = season_stats["goals"] / season_stats["minutes"] * 90
season_stats["assists_per_90"] = season_stats["assists"] / season_stats["minutes"] * 90

season_stats = season_stats.reset_index()

# Join player metadata
season_stats = season_stats.merge(
    players[["player_id", "name", "position", "date_of_birth"]],
    on="player_id",
    how="inner"
)

# Cast season to int for consistent merging
season_stats["season"] = season_stats["season"].astype(int)

# Join end-of-season market valuations
season_stats = season_stats.merge(end_of_season_value, on=["player_id", "season"], how="inner")

# Drop the ~50 rows with missing date_of_birth (age cannot be computed without it)
season_stats = season_stats.dropna(subset=["date_of_birth"])

# Calculate player age at end of season
season_stats["age"] = (
    (season_stats["season_end"] - season_stats["date_of_birth"]).dt.days / 365.25
).astype(int)

# Sanity check — Bukayo Saka (player_id 433177)
season_stats[season_stats['player_id'] == 433177]

## 5. Finalise and Save

Drop intermediate columns used only for calculation (`date_of_birth`, `season_end`) and save the clean dataset to `data/interim/` as a Parquet file.

The final dataset contains **93,133 rows** and **11 columns**, covering players across all seasons available in the database.

In [ ]:
df = season_stats.drop(columns=["date_of_birth", "season_end"])
df

In [ ]:
df.to_parquet("../data/interim/player_season_base.parquet", index=False)